In [ ]:
import csv
import json


# Step 1: Read the appointment CSV file

with open('appointments.csv', 'r', newline='') as file:

    reader = csv.DictReader(file)

    rows = list(reader)

    print("Header:", reader.fieldnames)
    print("Total rows:", len(rows))


# Step 2: Create parse_row function

def parse_row(row):

    row = row.copy()

    row['age'] = int(row['age'])
    row['fee_inr'] = int(row['fee_inr'])
    row['prior_no_shows'] = int(row['prior_no_shows'])
    row['distance_km'] = float(row['distance_km'])

    return row


# Step 3: Create accepted and rejected lists

accepted = []
rejected = []


# Step 4: Check every row

for row in rows:

    try:

        new_row = parse_row(row)

        accepted.append(new_row)

    except ValueError:

        rejected.append(row)


# Step 5: Print results

print("\nTotal rows:", len(rows))
print("Accepted rows:", len(accepted))
print("Rejected rows:", len(rejected))


# Step 6: Save rejected rows in JSON file

with open('rejected_rows.json', 'w') as file:

    json.dump(rejected, file, indent=2)


print("\nRejected rows saved successfully!")

In [ ]:
print("Insight 1 — Data quality")
print("32 records were rejected because some required numeric fields were blank. This shows that the original appointment dataset contains incomplete information.\n")

print("Insight 2 — Age information")
print("12 records have blank age values. Missing age information can make patient demographic analysis less reliable.\n")

print("Insight 3 — Distance information")
print("20 records have blank distance values. Missing distance information can affect analysis of the relationship between patient travel distance and appointment attendance.\n")

print("Insight 4 — Type validation versus business validation")
print("Five records have impossible ages, such as 250, but are accepted by Task 1.1 because they are numeric. This demonstrates that checking data types is different from checking whether values are logically valid.\n")




In [ ]:

import pandas as pd

# Step 1.2

# Step A: Load dataset
df = pd.read_csv('appointments.csv')

print("Original shape:", df.shape)


# Step B: Inspect dataset
print("\nDataset information:")
df.info()

print("\nMissing values:")
print(df.isnull().sum())

print("\nDuplicate rows:")
print(df.duplicated().sum())


# Step C: Remove duplicates
df = df.drop_duplicates()

print("\nRows after removing duplicates:", len(df))


# Step D: Clean booking_channel
df['booking_channel'] = (
    df['booking_channel'].str.strip().str.title()
)

print("\nUnique booking channels:")
print(df['booking_channel'].unique())

print("Number of channels:", df['booking_channel'].nunique())


# Step E: Convert dates
df['appt_date'] = pd.to_datetime(
    df['appt_date'], errors='coerce'
)

df['booked_on'] = pd.to_datetime(df['booked_on'])

print("\nDates converted successfully!")


# Step F: Remove missing appointment dates and status
df = df.dropna(subset=['appt_date', 'status'])

print("Final rows:", len(df))
print("Final shape:", df.shape)


In [ ]:
print("Insight 1 — Duplicate appointments")
print("20 duplicate rows were identified. Removing them prevents duplicate records from influencing appointment counts and no-show analysis.\n")

print("Insight 2 — Booking channel standardization")
print("The original 16 spellings were standardized into three categories: App, Phone and Walk-In. This makes channel-wise analysis more reliable.\n")

print("Insight 3 — Invalid appointment dates")
print("Six unreadable appointment dates were converted to missing values and removed. Invalid dates cannot reliably support appointment scheduling analysis.\n")

print("Insight 4 — Missing appointment status")
print("Four records had missing appointment status and were removed. Without a valid status, attendance classification cannot be performed correctly.")


In [ ]:
# Step 1.3

import pandas as pd
import numpy as np

# STEP A: Find invalid ages
invalid_age = (df['age'] < 1) | (df['age'] > 100)

print("Invalid ages:", invalid_age.sum())

df.loc[
    (df['age'] < 1) | (df['age'] > 100),
    'age'
] = np.nan


# STEP B: Fill missing values with median
age_median = df['age'].median()
distance_median = df['distance_km'].median()

df['age'] = df['age'].fillna(age_median)
df['distance_km'] = df['distance_km'].fillna(distance_median)

df['age'] = df['age'].round().astype(int)

print("Age median:", age_median)
print("Distance median:", distance_median)


# STEP C1: Cap distance outliers
Q1 = df['distance_km'].quantile(0.25)
Q3 = df['distance_km'].quantile(0.75)

IQR = Q3 - Q1

lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR

distance_count = (
    (df['distance_km'] < lower) |
    (df['distance_km'] > upper)
).sum()

df['distance_km'] = np.clip(
    df['distance_km'], lower, upper
)

print("Distance values capped:", distance_count)


# STEP C2: Cap fee outliers
Q1 = df['fee_inr'].quantile(0.25)
Q3 = df['fee_inr'].quantile(0.75)

IQR = Q3 - Q1

lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR

fee_count = (
    (df['fee_inr'] < lower) |
    (df['fee_inr'] > upper)
).sum()

df['fee_inr'] = np.clip(
    df['fee_inr'], lower, upper
)

print("Fee values capped:", fee_count)


# STEP D: Round values


df['distance_km'] = df['distance_km'].fillna(distance_median)
df['distance_km'] = df['distance_km'].round(1)

df['fee_inr'] = df['fee_inr'].round().astype(int)


# FINAL VERIFICATION
print("\nFinal missing values:")
print(df[['age', 'distance_km', 'fee_inr']].isnull().sum())

print("\nFinal dataset shape:", df.shape)

print("\nFirst five rows:")
print(df[['age', 'distance_km', 'fee_inr']].head())


In [ ]:
# Step 1.4

import numpy as np
import pandas as pd

# 1. Calculate lead_days
df['lead_days'] = (
    df['appt_date'] - df['booked_on']
).dt.days

print("1. Lead Days:")
print(df[['booked_on', 'appt_date', 'lead_days']].head())


# 2. Create no_show column
df['no_show'] = (
    df['status'] == 'No-Show'
).astype(int)

print("\n2. No-Show Distribution:")
print(df['no_show'].value_counts())


# 3. Create numerical feature matrix X
X = df[
    ['age', 'distance_km', 'fee_inr', 'lead_days']
].to_numpy()

print("\n3. Feature Matrix:")
print(X[:5])
print("Shape:", X.shape)


# 4. Calculate feature means
feature_means = np.mean(X, axis=0)

print("\n4. Feature Means:")
print(feature_means)


# 5. Analyse lead_days > 14
long_lead_mask = df['lead_days'] > 14

long_lead_count = long_lead_mask.sum()

long_lead_percentage = (
    long_lead_count / len(df)
) * 100

print("\n5. Lead Days > 14:")
print("Count:", long_lead_count)
print("Percentage:", round(long_lead_percentage, 2), "%")


# 6. Calculate distance Z-scores
distance_mean = np.mean(df['distance_km'])

distance_std = np.std(df['distance_km'])

df['distance_zscore'] = (
    df['distance_km'] - distance_mean
) / distance_std

zscore_outliers = (
    np.abs(df['distance_zscore']) > 2
).sum()

print("\n6. Distance Z-score Analysis:")
print("Mean:", distance_mean)
print("Standard Deviation:", distance_std)
print("Outliers:", zscore_outliers)



In [ ]:
import os

print(os.listdir('/content'))

In [ ]:

# Step 1.5

import requests
import pandas as pd
import time

# =====================================
# STEP 1: LOAD CLINIC DATASET
# =====================================

clinics_df = pd.read_csv("clinics.csv")

print("Original Clinic Dataset:")
print(clinics_df.head())

print("Total Clinics:", len(clinics_df))


# =====================================
# STEP 2: CREATE API FUNCTION
# =====================================

def get_temperature(latitude, longitude):

    url = "https://api.open-meteo.com/v1/forecast"

    params = {
        "latitude": latitude,
        "longitude": longitude,
        "hourly": "temperature_2m",
        "forecast_days": 1
    }

    for attempt in range(2):

        try:
            response = requests.get(
                url,
                params=params,
                timeout=10
            )

            if response.status_code != 200:
                raise requests.RequestException(
                    f"HTTP Status: {response.status_code}"
                )

            data = response.json()

            hourly = data.get("hourly", {})
            temperatures = hourly.get("temperature_2m")

            if not temperatures:
                raise ValueError("Hourly temperature data missing")

            temperature = temperatures[0]

            if temperature is None:
                raise ValueError("Temperature value is missing")

            return temperature

        except (requests.RequestException, ValueError, KeyError) as error:

            if attempt == 0:
                print("Request failed. Retrying...")
                time.sleep(1)

            else:
                print("API Error:", error)
                return None


# =====================================
# STEP 3: FETCH TEMPERATURE
# =====================================

temperatures = []

for index, row in clinics_df.iterrows():

    temperature = get_temperature(
        row['latitude'],
        row['longitude']
    )

    temperatures.append(temperature)

    if temperature is None:
        print(
            f"Failed to fetch temperature for clinic: "
            f"{row['clinic_id']}"
        )
    else:
        print(
            f"Clinic {row['clinic_id']}: "
            f"{temperature} °C"
        )

    time.sleep(0.2)


# =====================================
# STEP 4: ADD TEMP_C COLUMN
# =====================================

clinics_df['temp_c'] = temperatures

print("\nEnriched Clinic Dataset:")
print(clinics_df)


# =====================================
# STEP 5: SAVE ENRICHED DATASET
# =====================================

clinics_df.to_csv(
    "clinics_enriched.csv",
    index=False
)

print("\nclinics_enriched.csv saved successfully!")


In [ ]:

import pandas as pd

# STEP A: Define required columns
columns = [
    'appointment_id',
    'patient_id',
    'clinic_id',
    'booked_on',
    'appt_date',
    'age',
    'gender',
    'distance_km',
    'fee_inr',
    'booking_channel',
    'visit_type',
    'reminder_sent',
    'prior_no_shows',
    'status',
    'lead_days',
    'no_show'
]


# STEP B: Sort appointment data
df = df.sort_values('appointment_id')


# STEP C: Check appointment IDs
print("Unique IDs:", df['appointment_id'].nunique())
print("Total rows:", len(df))

print("\nMissing values:")
print(df[columns].isnull().sum())


# STEP D: Select required columns
clean_df = df[columns].copy()


# STEP E: Format dates
clean_df['booked_on'] = pd.to_datetime(
    clean_df['booked_on']
).dt.strftime('%Y-%m-%d')

clean_df['appt_date'] = pd.to_datetime(
    clean_df['appt_date']
).dt.strftime('%Y-%m-%d')


# STEP F: Save clean CSV
clean_df.to_csv('appointments_clean.csv', index=False)

print("\nClean CSV saved!")
print("Final shape:", clean_df.shape)


# STEP G: Read both CSV files
clinics_sql = pd.read_csv('clinics_enriched.csv')

appointments_sql = pd.read_csv('appointments_clean.csv')


# STEP H: Create SQL value function
def sql_value(value, column, numeric_columns):

    if pd.isna(value) or str(value).strip() == '':
        return 'NULL'

    if column in numeric_columns:
        return str(value)

    return "'" + str(value).replace("'", "''") + "'"


# STEP I: Define numeric columns
clinic_numbers = ['latitude', 'longitude', 'temp_c']

appointment_numbers = [
    'age',
    'distance_km',
    'fee_inr',
    'prior_no_shows',
    'lead_days',
    'no_show'
]


# STEP J: Create SQL file
with open('load_data.sql', 'w') as file:

    # Insert clinics first
    for index, row in clinics_sql.iterrows():

        values = []

        for column in clinics_sql.columns:

            value = sql_value(
                row[column], column, clinic_numbers
            )

            values.append(value)

        file.write(
            f"INSERT INTO clinics VALUES ({','.join(values)});\n"
        )


    # Insert appointments second
    for index, row in appointments_sql.iterrows():

        values = []

        for column in appointments_sql.columns:

            value = sql_value(
                row[column], column, appointment_numbers
            )

            values.append(value)

        file.write(
            f"INSERT INTO appointments VALUES ({','.join(values)});\n"
        )


# STEP K: Final verification
print("\nClinic INSERT statements:", len(clinics_sql))

print("Appointment INSERT statements:", len(appointments_sql))

print("Total INSERT statements:",
      len(clinics_sql) + len(appointments_sql))

print("\nAll files created successfully!")
